# 01 Data Understanding

This notebook documents the data-understanding phase for the AML graph-enhanced data mining project.

Main goals:
- Load IBM AMLworld `HI-Small_Trans.csv`.
- Inspect schema, class imbalance, timestamps, transaction fields, and graph structure.
- Create report-ready label-distribution and timeline artifacts.
- Keep the output aligned with the final project scope: transaction-level AML scoring using graph-enhanced machine learning.

In [ ]:
from pathlib import Path
import sys

# Works whether the notebook is launched from repository root or from notebooks/.
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD.parent if CWD.name == "notebooks" else CWD

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"
TABLES_DIR = REPORTS_DIR / "tables"
FIGURES_DIR = REPORTS_DIR / "figures"
MODELS_DIR = PROJECT_ROOT / "models"

for path in [PROCESSED_DIR, TABLES_DIR, FIGURES_DIR, MODELS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)


import pandas as pd
import matplotlib.pyplot as plt

from src.data.load_data import load_transactions
from src.features.graph_features import build_sample_graph, graph_summary

## 1. Configuration

Use `NROWS = 2_000_000` to match the final experiment setup. Set `NROWS = None` only when you intentionally want to inspect the full HI-Small file.

In [ ]:
DATASET_NAME = "HI-Small"
RAW_PATH = RAW_DIR / f"{DATASET_NAME}_Trans.csv"
NROWS = 2_000_000

if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Raw dataset not found: {RAW_PATH}\n"
        "Put HI-Small_Trans.csv inside data/raw/ before running this notebook."
    )

df = load_transactions(RAW_PATH, nrows=NROWS)
display(df.head())
print("Loaded shape:", df.shape)

## 2. Schema and target distribution

In [ ]:
print("Schema:")
display(df.dtypes)

print("\nTarget counts:")
label_counts = df["is_laundering"].value_counts(dropna=False).sort_index()
display(label_counts)

print("\nTarget rates:")
label_rates = df["is_laundering"].value_counts(normalize=True, dropna=False).sort_index()
display(label_rates)

summary = pd.DataFrame({
    "count": label_counts,
    "rate": label_rates,
})
summary.index = summary.index.map({0: "normal", 1: "laundering"})
display(summary)

## 3. Label distribution figure

In [ ]:
label_plot = df["is_laundering"].value_counts().sort_index()
ax = label_plot.plot(kind="bar")
ax.set_title("Label Distribution")
ax.set_xlabel("Is Laundering")
ax.set_ylabel("Transaction Count")
ax.set_xticklabels(["Normal", "Laundering"], rotation=0)
plt.tight_layout()

out_path = FIGURES_DIR / "label_distribution.png"
plt.savefig(out_path, dpi=150)
plt.show()

print("Saved:", out_path)

## 4. Transaction field overview

In [ ]:
for col in ["payment_format", "payment_currency", "receiving_currency", "from_bank", "to_bank"]:
    if col in df.columns:
        print(f"\nTop values for {col}:")
        display(df[col].value_counts(dropna=False).head(10))

## 5. Timestamp coverage

In [ ]:
if "timestamp" in df.columns:
    print("Minimum timestamp:", df["timestamp"].min())
    print("Maximum timestamp:", df["timestamp"].max())
    print("Number of unique timestamps:", df["timestamp"].nunique())

    daily = (
        df.assign(date=df["timestamp"].dt.date)
        .groupby("date")
        .agg(
            rows=("is_laundering", "size"),
            laundering_count=("is_laundering", "sum"),
            laundering_rate=("is_laundering", "mean"),
        )
        .reset_index()
    )
    display(daily.head())
    display(daily.tail())

    daily.to_csv(TABLES_DIR / "daily_label_distribution_sample.csv", index=False)
    print("Saved:", TABLES_DIR / "daily_label_distribution_sample.csv")

## 6. Label timeline audit by row chunks

In [ ]:
timeline_path = TABLES_DIR / "hi_small_label_timeline.csv"

if timeline_path.exists():
    timeline = pd.read_csv(timeline_path)
    display(timeline)
else:
    print(
        "Timeline file not found. To create it, run:\n"
        "PYTHONPATH=. python scripts/audit_label_timeline.py --dataset HI-Small --chunksize 250000"
    )

## 7. Sample graph statistics

In [ ]:
# Build only a sample graph for quick exploratory graph statistics.
# Full graph feature engineering is handled in 02_feature_engineering.ipynb and src/features/.
G = build_sample_graph(df, max_edges=50_000)
graph_stats = graph_summary(G)

display(pd.DataFrame([graph_stats]))
pd.DataFrame([graph_stats]).to_csv(TABLES_DIR / "sample_graph_summary.csv", index=False)
print("Saved:", TABLES_DIR / "sample_graph_summary.csv")